In [ ]:
# %%
import math, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs("results/figs", exist_ok=True)
plt.rcParams["figure.figsize"] = (9, 5)
QUICK = False
N_SEEDS = 40 if QUICK else 200

# ---- same world as the notebook (do not change, or results will not match) ----
_layout = np.random.default_rng(7)
N_NODES = 12
XY = _layout.uniform(0, 900, size=(N_NODES, 2))
SPEED = 130.0
T = np.ceil(np.linalg.norm(XY[:, None] - XY[None], axis=2) / SPEED).astype(int)
N_BINS = N_NODES - 1
BASE_RATE = _layout.uniform(20, 48, size=N_BINS)
DAY_MIN, START_HOUR = 720, 8


def demand_curve(h):
    return (0.25 + 1.2 * math.exp(-((h - 12.5) / 1.2) ** 2)
            + 0.6 * math.exp(-((h - 10.0) / 0.6) ** 2)
            + 0.6 * math.exp(-((h - 15.0) / 0.7) ** 2))


CURVE = np.array([demand_curve(START_HOUR + t / 60) for t in range(DAY_MIN)])


def fill_paths(seed, scale=1.0):
    """Fill level (%) of every bin every minute with NO robots: shape (DAY_MIN, N_BINS).
    Uses the same RNG draw order as simulate() so seed k is the same day."""
    rng = np.random.default_rng(seed)
    day_mult = rng.lognormal(0, 0.2, N_BINS)
    noise = np.clip(1 + 0.3 * rng.standard_normal((DAY_MIN, N_BINS)), 0, None)
    start = rng.uniform(0, 50, N_BINS)
    inc = BASE_RATE * day_mult * CURVE[:, None] * scale * noise / 60
    return np.minimum(start + np.cumsum(inc, axis=0), 100.0), day_mult


PATHS = [fill_paths(s)[0] for s in range(N_SEEDS)]
DAYMULT = np.array([fill_paths(s)[1] for s in range(N_SEEDS)])
print(f"{N_SEEDS} simulated days, {N_BINS} bins")

# %% [markdown]
# ## E. Simulation validation  (sim.py)
# Needs results/runs_main.csv from the fleet-sizing notebook. Answers: how many days
# are enough, and is the policy gap bigger than day-to-day noise?

# %%
CSV = "results/runs_main.csv"
if not os.path.exists(CSV):
    print(f"\n[E skipped] {CSV} not found. Run the fleet-sizing notebook first.")
else:
    runs = pd.read_csv(CSV)
    K = 5 if 5 in set(runs.robots) else int(sorted(set(runs.robots))[len(set(runs.robots)) // 2])
    d = runs[(runs.robots == K) & (runs.demand_scale == 1.0)]
    print(f"\nUsing {K} robots, demand 1.0x")

    # 1) how many seeds for a stable mean? half-width of the 95% CI vs n
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
    need = {}
    for pol, g in d.groupby("policy"):
        x = g.sort_values("seed").avg_wait_min.values
        ns = np.arange(2, len(x) + 1)
        hw = [1.96 * x[:n].std(ddof=1) / math.sqrt(n) for n in ns]
        ax[0].plot(ns, hw, label=pol)
        s = x.std(ddof=1)
        need[pol] = math.ceil((1.96 * s / 0.25) ** 2)         # days for +/-0.25 min precision
    ax[0].axhline(0.25, ls="--", color="gray")
    ax[0].set(xlabel="number of simulated days", ylabel="95% CI half-width of avg wait (min)",
              title="Precision vs days (dashed = +/-0.25 min)"); ax[0].legend()
    print("Days needed for +/-0.25 min precision on avg wait:", need)

    # 2) paired sign-flip permutation test vs greedy (no scipy needed)
    base = d[d.policy == "greedy"].set_index("seed")
    out = []
    rng = np.random.default_rng(0)
    for pol in sorted(set(d.policy) - {"greedy"}):
        o = d[d.policy == pol].set_index("seed")
        for col in ("avg_wait_min", "max_wait_min", "overflow_minutes"):
            diff = (o[col] - base[col]).dropna().values
            obs = diff.mean()
            flips = rng.choice([-1, 1], size=(20000, len(diff))) * diff
            p = (np.abs(flips.mean(axis=1)) >= abs(obs) - 1e-12).mean()
            out.append({"policy": pol, "metric": col, "mean_diff": obs, "perm_p": p,
                        "sd_of_diff": diff.std(ddof=1),
                        "cohens_dz": obs / diff.std(ddof=1) if diff.std(ddof=1) > 0 else np.nan})
    tests = pd.DataFrame(out)
    tests.to_csv("results/paired_tests.csv", index=False)
    print(tests.round(3).to_string(index=False))

    # 3) share of days meeting the 15-min target with Wilson interval (honest for small n)
    def wilson(k, n, z=1.96):
        p = k / n; den = 1 + z * z / n
        c = (p + z * z / (2 * n)) / den
        h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
        return max(0, c - h), min(1, c + h)

    pols = sorted(set(d.policy))
    for j, pol in enumerate(pols):
        g = d[d.policy == pol]
        k, n = int((g.max_wait_min <= 15).sum()), len(g)
        lo, hi = wilson(k, n)
        ax[1].errorbar(j, k / n, yerr=[[k / n - lo], [hi - k / n]], fmt="o", capsize=5)
        print(f"  {pol:10s} meets 15-min target on {k}/{n} days; 95% Wilson CI [{lo:.2f}, {hi:.2f}]")
    ax[1].set_xticks(range(len(pols))); ax[1].set_xticklabels(pols)
    ax[1].set(ylabel="share of days meeting target", ylim=(0, 1.05), title="Service level with honest intervals")
    plt.tight_layout(); plt.savefig("results/figs/E_sim_validation.png", dpi=130); plt.show()

print("\nDone. Figures in results/figs/, tables in results/.")